# Koszt obliczeniowy komponentów

Czyta gotowe pliki pomiaru `results/measurements/cost_*.json` i układa z nich tabele kosztu: czas ekstrakcji, pamięć karty, miejsce na dysku, czas obsługi zapytania. Sam niczego nie mierzy. Każda sekcja brana jest z najnowszego pliku, który ją zawiera, więc domiar jednego komponentu nie unieważnia reszty pomiaru.

**Wymaga:** pomiaru kosztu. Przed uruchomieniem zamknij wszystko, co zajmuje kartę - LLaVA chce 14 GB, a przeglądarka trzymająca pół giga zmienia mierzony szczyt pamięci.

```powershell
python scripts/measure_cost.py --dataset office --split test
```

Pomiar trwa około pół godziny. Ekstrakcja mierzy się na próbce rozłożonej po odcinkach splitu: 320 klatek siatki i 512 wyrównanych wycinków twarzy, każde wywołanie piętnaście razy po trzech rozgrzewkowych, z czasów mediana. O długości przebiegu decyduje niemal wyłącznie LLaVA (sekunda na klatkę razy 64 klatki razy 18 przebiegów), więc `--caption-frames` skraca albo zacieśnia cały pomiar.

Bez `--config` mierzona jest konfiguracja potoku pełnego, `configs/full_<zbiór>.yaml`. Tylko konfiguracja z włączonymi komponentami ma dla nich czas obsługi zapytania, inaczej kolumna "Zapytanie" wypełniłaby się wyłącznie dla enkodera bazowego. Pierwszy przebieg doliczy przy okazji to, czego pełny potok nie ma jeszcze na dysku, najpewniej bufor ruchu SlowFast.

Przełączniki: `--skip-extraction` zostawia sam odczyt rozmiarów katalogów i fazę zapytania i nie ładuje żadnego modelu, `--skip-query` odwrotnie, `--only <komponent>` zawęża ekstrakcję do wybranych modeli (`openclip_vit_h14`, `openclip_vit_b32`, `xclip`, `slowfast`, `blip`, `llava`, `yolo11`, `yoloe`, `retinaface`, `arcface`, `hsemotion`).

**Zapisuje:** nic, tylko wypisuje.

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path
from statistics import mean

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import tables
from src.utils import experiments as exp

MEASUREMENTS = ROOT / "results" / "measurements"

#: the part the cost table rests on: the test measurement of every series
SPLIT = "test"

#: above this spread of the per-unit time the two series disagree about a
#: component rather than about the state of the machine. Five per cent, because
#: the within-run spread of every component but one sits below it.
SERIES_TOLERANCE = 0.05


def newest_with(section):
    """Newest measurement file that actually carries this section.

    Used only by the E4-D block, which has a single measurement of its own. The
    main table does NOT work this way -- see series_measurements.
    """
    for path in sorted(MEASUREMENTS.glob("cost_*.json"), reverse=True):
        record = json.loads(path.read_text(encoding="utf-8"))
        if record["data"].get(section):
            return path, record
    return None


def amount(value):
    """Enough decimals that the small components do not collapse to one value."""
    if value is None:
        return "-"
    size = abs(value)
    if size == 0:
        return "0"
    return tables.number(value, 1 if size >= 10 else 2 if size >= 0.01 else 3)


def series_measurements(split=SPLIT):
    """``{dataset: (path, record)}`` -- the cost measurement of every series.

    The table reports the MEAN over the series, not the newest file. The script
    writes one file per dataset, so "newest wins" silently reported ONE series --
    and the two disagree by more than a fifth on the caption generator, which is
    the most expensive component of the pipeline. A rate per hour of material is
    exactly the kind of number that has no business depending on which file was
    written last.
    """
    found = {}
    for path in sorted(MEASUREMENTS.glob("cost_*.json")):
        record = json.loads(path.read_text(encoding="utf-8"))
        data = record["data"]
        if data.get("split") != split or data.get("dataset") not in exp.SERIES:
            continue
        found[data["dataset"]] = (path, record)
    return found


def mean_of(values):
    """Mean of the values that are there; None when none of them is."""
    present = [value for value in values if value is not None]
    return sum(present) / len(present) if present else None


def average_section(records, section, identity, numeric):
    """Entries of one section, numeric fields averaged over the datasets.

    Returns ``(by_id, order, datasets_of)``. A component measured on only ONE
    dataset keeps that dataset's numbers and is named in ``datasets_of``, so the
    block can say so out loud instead of printing an average of one and calling
    it the same kind of number as the rest.
    """
    order, groups = [], {}
    for dataset, (_, record) in sorted(records.items()):
        for entry in record["data"].get(section) or []:
            name = entry[identity]
            if name not in groups:
                groups[name] = []
                order.append(name)
            groups[name].append((dataset, entry))
    by_id, datasets_of = {}, {}
    for name in order:
        merged = dict(groups[name][0][1])
        for field in numeric:
            merged[field] = mean_of([entry.get(field) for _, entry in groups[name]])
        by_id[name] = merged
        datasets_of[name] = sorted(dataset for dataset, _ in groups[name])
    return by_id, order, datasets_of


def average_query(records):
    """The query section averaged: a row per signal, plus phrases and fusion.

    The COMPOSITION is a property of the configuration and the same in both, so
    it is taken whole; the collection size is a property of the material and
    differs, so both values travel and the table says so.
    """
    parts = {dataset: record["data"]["query"]
             for dataset, (_, record) in sorted(records.items())
             if record["data"].get("query")}
    if not parts:
        return {}, {}
    order, groups = [], {}
    for dataset, part in parts.items():
        for entry in part.get("signals", []):
            name = entry["signal"]
            if name not in groups:
                groups[name] = []
                order.append(name)
            groups[name].append((dataset, entry))

    merged = {"signals": [
        {"signal": name,
         "median_ms": mean_of([e["median_ms"] for _, e in groups[name]]),
         "p95_ms": mean_of([e["p95_ms"] for _, e in groups[name]])}
        for name in order]}
    for name in ("phrases", "fusion"):
        found = [part[name] for part in parts.values() if part.get(name)]
        if found:
            merged[name] = {"median_ms": mean_of([e["median_ms"] for e in found]),
                            "p95_ms": mean_of([e["p95_ms"] for e in found])}
    first = parts[sorted(parts)[0]]
    merged["components"] = first.get("components")
    merged["config"] = {dataset: part.get("config") for dataset, part in parts.items()}
    merged["collection"] = {dataset: part.get("collection")
                            for dataset, part in parts.items()}
    merged["queries"] = {dataset: part.get("queries") for dataset, part in parts.items()}
    return merged, {name: sorted(d for d, _ in groups[name]) for name in order}


#: fields averaged in each section; everything else is carried over whole
EXTRACTION_NUMERIC = ("seconds_per_unit", "min_per_hour", "peak_vram_gb",
                      "torch_vram_gb", "vram_noise_gb", "load_s", "spread")
INDEX_NUMERIC = ("mb", "mb_per_hour")

records = series_measurements()
extraction, extraction_keys, extraction_on = average_section(
    records, "extraction", "key", EXTRACTION_NUMERIC)
index, index_names, index_on = average_section(
    records, "index", "component", INDEX_NUMERIC)
query, query_on = average_query(records)
extraction_order = [extraction[key] for key in extraction_keys]

# the per-series numbers stay reachable: the extraction table shows both, and
# the averaged table has to be checkable against what it was averaged from
per_series = {dataset: {"extraction": {e["key"]: e
                                       for e in record["data"].get("extraction") or []},
                        "index": {e["component"]: e
                                  for e in record["data"].get("index") or []},
                        "query": record["data"].get("query") or {},
                        "hours": record["data"].get("corpus_hours"),
                        "file": path.name}
              for dataset, (path, record) in sorted(records.items())}

machine = next((record["machine"] for _, record in records.values()), {})
meta = next((record["data"] for _, record in records.values()), {})
strategies = {record["data"].get("strategy") for _, record in records.values()}
strategy = next(iter(strategies)) if len(strategies) == 1 else None

if not records:
    print(f"brak pomiaru na czesci {SPLIT!r} - uruchom scripts/measure_cost.py")
else:
    print(f"tabela usredniona z {len(records)} pomiarow (czesc {SPLIT!r}):")
    for dataset, (path, record) in sorted(records.items()):
        print(f"  {path.name:<30}{exp.DATASET_NAMES.get(dataset, dataset)}"
              f", {record['data'].get('corpus_hours')} h korpusu")

    # A section present in only one file is not an average and must not be read
    # as one. Reported per section AND per component, because a --only run tops
    # up single components and leaves the rest of the section where it was.
    for label, present, total in (("ekstrakcji", extraction_on, len(records)),
                                  ("indeksu", index_on, len(records)),
                                  ("zapytania", query_on, len(records))):
        lonely = {name: where for name, where in present.items() if len(where) < total}
        if lonely:
            print(f"\nUWAGA: te pozycje sekcji {label} pochodza z jednego zbioru, "
                  "wiec NIE sa srednia:")
            for name, where in sorted(lonely.items()):
                print(f"  {name:<28}{', '.join(exp.DATASET_NAMES.get(d, d) for d in where)}")
    missing_section = [label for label, block in (("ekstrakcja", extraction),
                                                  ("indeks", index),
                                                  ("zapytanie", query)) if not block]
    if missing_section:
        print(f"\nUWAGA: brak sekcji {', '.join(missing_section)} w obu pomiarach")

    print(f"\n  karta:        {machine.get('gpu')}")
    print(f"  segmentacja:  {strategy or 'ROZNA w pomiarach - sprawdz'}")
    print(f"  powtorzenia:  {meta.get('repeats')} (+{meta.get('warmup')} rozgrzewki)")
    print(f"  plikow pomiaru w katalogu: {len(list(MEASUREMENTS.glob('cost_*.json')))}")

## 1. Koszt obliczeniowy komponentów

Jeden wiersz na komponent, w kolejności, w jakiej wymienia je praca.

Kolumna **Indeks** obejmuje wszystko, co dany komponent zostawia na dysku, także reprezentacje trzymane poza indeksem wektorowym (katalogi opisów, bufor wycinków twarzy, rozkłady klas). Kolumna **Zapytanie** podaje medianę i w nawiasie 95. percentyl; wypełnia się tylko dla komponentów włączonych w konfiguracji, na której mierzono fazę zapytania.

Myślnik znaczy "nie mierzono", zero - "zmierzono i wyszło poniżej rozdzielczości zapisu".

In [ ]:
# label -> (extraction key, index name); None means the quantity does not exist
ROWS = [
    ("OpenCLIP ViT-H/14",           "openclip_vit_h14", "OpenCLIP ViT-H/14"),
    ("OpenCLIP ViT-B/32",           "openclip_vit_b32", "OpenCLIP ViT-B/32"),
    ("X-CLIP",                      "xclip",            "X-CLIP"),
    ("SlowFast",                    "slowfast",         "SlowFast"),
    ("BLIP",                        "blip",             "BLIP"),
    ("LLaVA-1.5",                   "llava",            "LLaVA-1.5"),
    ("YOLO11",                      "yolo11",           "YOLO11"),
    ("YOLOE-11",                    "yoloe",            "YOLOE-11"),
    ("RetinaFace",                  "retinaface",       "RetinaFace (crop buffer)"),
    ("ArcFace",                     "arcface",          "ArcFace"),
    ("regiony twarzy",              "regions",          "face regions"),
    ("HSEmotion",                   "hsemotion",        "HSEmotion"),
]

# which row carries the query time of a signal; the variant-dependent ones are
# resolved from the configuration the query phase was measured on
# the keys are the values the configuration schema allows, verbatim: they are
# looked up with the variant read from the measured configuration, so a name that
# only looks right ("llava" for llava_1_5_7b) silently drops the row's query time
SIGNAL_ROW = {"scene_embedding": {"openclip_vit_h14": "OpenCLIP ViT-H/14",
                                  "openclip_vit_b32": "OpenCLIP ViT-B/32",
                                  "xclip_b32": "X-CLIP"},
              "caption": {"blip": "BLIP", "llava_1_5_7b": "LLaVA-1.5"},
              "objects": {"yolo11": "YOLO11", "yoloe_promptfree": "YOLOE-11"},
              "face_regions": {"clip_regions": "regiony twarzy",
                               "hsemotion": "HSEmotion"},
              "identity": {None: "ArcFace"},
              "motion": {None: "SlowFast"}}


def enabled_components(query):
    """{component: settings} of the measured configuration, as the run recorded it.

    Read from the measurement file, not from `configs/`: the record has to stay
    readable after its configuration file is renamed, reworked or removed.
    """
    return (query or {}).get("components") or {}


def row_variant(signal, settings):
    """The key under which SIGNAL_ROW holds the row label of one signal.

    A signal with a single possible row keys it under None -- the row does not
    depend on the settings. Reading a model name for such a signal returns a key
    that is not in the map, and the row drops out without a word: that is how the
    motion row went missing from the totals once.
    """
    if set(SIGNAL_ROW[signal]) == {None}:
        return None
    return (settings.get("model") or settings.get("detector")
            or settings.get("mode"))


def query_rows(query):
    """Row labels the measured configuration actually uses.

    RetinaFace joins them whenever a face signal is on: it has no signal of its
    own, but both the regions and the identity read its crop buffer.
    """
    components = enabled_components(query)
    labels = []
    for signal in SIGNAL_ROW:
        settings = components.get(signal)
        if settings is None:
            continue
        label = SIGNAL_ROW[signal].get(row_variant(signal, settings))
        if label:
            labels.append(label)
    if "face_regions" in components or "identity" in components:
        labels.append("RetinaFace")
    return labels


def query_by_row(query):
    """{row label: (median, p95)} for the signals of the measured configuration."""
    components = enabled_components(query)
    if not components:
        return {}
    variant = {signal: row_variant(signal, components.get(signal) or {})
               for signal in SIGNAL_ROW}
    out = {}
    for signal in query["signals"]:
        row = SIGNAL_ROW.get(signal["signal"], {}).get(variant.get(signal["signal"]))
        if row:
            out[row] = (signal["median_ms"], signal["p95_ms"])
    return out


if query and not query.get("components"):
    print("UWAGA: pomiar bez listy komponentow - kolumna 'Zapytanie'"
          " i wiersz RAZEM zostana puste;"
          " powtorz pomiar nowym measure_cost.py")

if extraction or index:
    timings = query_by_row(query)
    rows = []
    for label, key, index_name in ROWS:
        measured = extraction.get(key) if key else None
        size = index.get(index_name)
        pair = timings.get(label)
        rows.append([
            label,
            amount(measured["min_per_hour"]) if measured else "-",
            amount(measured["peak_vram_gb"]) if measured else "-",
            amount(size["mb_per_hour"]) if size else "-",
            f"{amount(pair[0])} ({amount(pair[1])})" if pair else "-"])

    # the text layer has no model, no index and no extraction, but it does have
    # a query cost, and it runs before every vocabulary signal: leaving it out
    # would make the column stop adding up to its own total
    phrases = (query or {}).get("phrases")
    rows.append(["frazy zapytania", "-", "-", "-",
                 f"{amount(phrases['median_ms'])} ({amount(phrases['p95_ms'])})"
                 if phrases else "-"])

    fusion = (query or {}).get("fusion")
    rows.append(["laczenie sygnalow i ranking", "-", "-", "-",
                 f"{amount(fusion['median_ms'])} ({amount(fusion['p95_ms'])})"
                 if fusion else "-"])

    # The table lists every measured component, but no pipeline runs all of them:
    # the variants of one decision are alternatives. The total therefore covers
    # only the composition actually measured in the query phase, read from its
    # configuration, so it cannot drift from the frozen one.
    used = set(query_rows(query))
    if used:
        by_label = {label: (key, index_name) for label, key, index_name in ROWS}
        chosen = [by_label[label] for label in used if label in by_label]
        extraction_total = sum(extraction[k]["min_per_hour"]
                               for k, _ in chosen if k and k in extraction)
        index_total = sum(index[n]["mb_per_hour"]
                          for _, n in chosen if n in index and index[n]["mb_per_hour"])
        # memory does NOT add up: components are loaded and released one at a
        # time, so what the card has to hold is the largest of them
        vram_peak = max((extraction[k]["peak_vram_gb"]
                         for k, _ in chosen if k and k in extraction
                         and extraction[k]["peak_vram_gb"]), default=None)
        query_total = sum(s["median_ms"] for s in query["signals"])
        query_total += phrases["median_ms"] if phrases else 0.0
        query_total += fusion["median_ms"] if fusion else 0.0
        rows.append(["RAZEM (potok PELNY)",
                     amount(extraction_total),
                     f"max {amount(vram_peak)}" if vram_peak else "-",
                     amount(index_total),
                     amount(query_total)])

    tables.show("Koszt obliczeniowy komponentow potoku "
                "(czas zapytania: mediana i 95. percentyl)",
                ["Komponent", "Ekstrakcja [min/h]", "Pamiec GPU [GB]",
                 "Indeks [MB/h]", "Zapytanie [ms]"], rows, align="lrrrr",
                note="Ekstrakcja i indeks znormalizowane na godzine materialu. "
                     "Regiony twarzy nie maja wlasnego modelu: wycinki koduje "
                     "enkoder bazowy. Ich wiersz jest mimo to osobny, bo wejscie "
                     "jest inne - enkoder chodzi tam po wycinkach twarzy, a w "
                     "swoim wierszu po klatkach siatki, i na godzine materialu jednych i drugich jest inaczej wiele. "
                     "RAZEM obejmuje wylacznie sklad zmierzonej konfiguracji - "
                     "warianty tej samej decyzji sie wykluczaja, wiec nie sumuja "
                     "sie. Pamiec nie sumuje sie nigdy: komponenty laduja sie "
                     "pojedynczo, wiec karta musi pomiescic najwiekszy z nich.")
else:
    print("brak pomiaru")

## 2. Powtarzalność pomiaru

Tabela wyżej pokazuje najnowszy pomiar każdego komponentu. Ta sekcja zestawia wszystkie przebiegi leżące w `results/measurements/` i mówi, na ile są powtarzalne.

Kolumna "Rozstęp" to `(max - min) / średnia` po przebiegach. Jeśli jest tego samego rzędu co rozrzut wewnątrz pojedynczego przebiegu (skrypt wypisuje go przy każdym komponencie), o niepewności decyduje stan maszyny, a nie wielkość próby, i dokładanie powtórzeń nic nie poprawi. Pomaga wtedy tylko zamknięcie wszystkiego, co zajmuje kartę.

In [ ]:
measured = defaultdict(list)
for path in sorted(MEASUREMENTS.glob("cost_*.json")):
    record = json.loads(path.read_text(encoding="utf-8"))
    for entry in record["data"].get("extraction", []):
        measured[entry["key"]].append({"file": path.name, **entry})

if measured:
    rows = []
    for label, key, _ in ROWS:
        group = measured.get(key or "")
        if not group:
            continue
        per_unit = [1000 * e["seconds_per_unit"] for e in group]
        vram = [e["peak_vram_gb"] for e in group if e["peak_vram_gb"]]
        sizes = sorted({e["batch"] for e in group})
        rows.append([
            label, len(group),
            amount(mean(per_unit)),
            f"{amount(min(per_unit))} - {amount(max(per_unit))}",
            amount(100 * (max(per_unit) - min(per_unit)) / mean(per_unit)) + "%",
            amount(mean([e["min_per_hour"] for e in group])),
            amount(mean(vram)) if vram else "-",
            " / ".join(str(s) for s in sizes)])

    tables.show("Powtarzalnosc: srednia ze wszystkich przebiegow",
                ["Komponent", "Przebiegow", "Sr. czas [ms/jedn.]", "Zakres",
                 "Rozstep", "Sr. ekstrakcja [min/h]", "Sr. pamiec [GB]", "Probka"],
                rows, align="lrrrrrrr",
                note="Srednia, nie mediana: przebiegow jest kilka, a mediana z "
                     "dwoch to i tak ich srednia, przy trzech zas gubi jeden "
                     "pomiar. Rozstep to (max-min)/srednia i mowi, na ile cyfr "
                     "mozna liczyc. Kolumna 'Probka' pokazuje, ile jednostek "
                     "obejmowal przebieg - wartosci z roznych probek nie sa "
                     "scisle porownywalne i srednia z nich jest orientacyjna.")
else:
    print("brak pomiarow ekstrakcji")

## 3. Czas obsługi zapytania

Mierzone na prawdziwej kolekcji, zapytanie po zapytaniu, po rozgrzaniu - stąd 95. percentyl mówi o rozrzucie, a nie o pierwszym, zimnym przebiegu.

In [ ]:
if query:
    # config and collection differ between the series -- one is the configuration
    # of a dataset, the other a property of its material -- so neither is averaged
    # into a single number that would belong to no collection at all
    for dataset, config in sorted((query.get("config") or {}).items()):
        print(f"konfiguracja {exp.DATASET_NAMES.get(dataset, dataset)}: {config}")
        print(f"  kolekcja: {query['collection'].get(dataset)} fragmentow, "
              f"{query['queries'].get(dataset)} zapytan")
    print("czasy ponizej sa srednia z obu przebiegow")

    # The phrase layer runs ONCE per query, before any closed-vocabulary signal:
    # spaCy parses the query, WordNet decides what each phrase is, the text encoder
    # embeds every form. Pipeline.signal_matrices counts it inside its per-query
    # timing, so it belongs in this table too - and in RAZEM, or the sum would
    # under-report what a query actually costs.
    phrases = query.get("phrases")
    rows = []
    if phrases:
        rows.append(["frazy zapytania", amount(phrases["median_ms"]),
                     amount(phrases["p95_ms"])])
    rows += [[s["signal"], amount(s["median_ms"]), amount(s["p95_ms"])]
             for s in query["signals"]]
    rows.append(["laczenie sygnalow i ranking",
                 amount(query["fusion"]["median_ms"]),
                 amount(query["fusion"]["p95_ms"])])
    total = sum(s["median_ms"] for s in query["signals"]) + query["fusion"]["median_ms"]
    if phrases:
        total += phrases["median_ms"]
    rows.append(["RAZEM", amount(total), "-"])

    tables.show("Czas obslugi zapytania",
                ["Skladnik", "Mediana [ms]", "95. percentyl [ms]"], rows,
                note="Kazdy sygnal mierzony osobno na tej samej kolekcji, na "
                     "frazach juz wydobytych - tak jak w przebiegu. Wiersz RAZEM "
                     "to suma median, wiec jest przyblizeniem: sygnaly dziela "
                     "enkoder zapytania i liczone razem sa tansze. Mierzone sa "
                     "tylko sygnaly WLACZONE w konfiguracji powyzej.")
    if phrases is None:
        print("\nUWAGA: pomiar bez wiersza 'frazy zapytania' - pochodzi sprzed "
              "dolozenia warstwy fraz do measure_cost.py. Czasy sygnalow "
              "slownikowych (obiekty, ruch, regiony) sa w nim zanizone, bo "
              "mierzono je bez fraz, czyli na sciezce, ktora wraca od razu. "
              "Powtorz pomiar.")
else:
    print("pomiar bez fazy zapytania - uruchom bez --skip-query")

## 4. Co składa się na czas ekstrakcji

Czas na jednostkę, wsad i czas ładowania modelu, osobno dla każdego serialu - to jedyne miejsce, w którym widać, czy uśrednienie w tabeli wyżej cokolwiek zaciera. Kolumna "Rozstęp" to `(max - min) / średnia` między serialami, kolumna "rozrzut" ten sam wskaźnik wewnątrz pojedynczego przebiegu. Gdy oba są tego samego rzędu, różnica między serialami jest szumem maszyny; gdy rozstęp jest wyraźnie większy, zależy od materiału. Pod tabelą blok wypisuje komponenty przekraczające próg `SERIES_TOLERANCE`.

In [ ]:
if len(per_series) < 2:
    print("rozbicie per serial wymaga pomiaru obu seriali - jest "
          f"{len(per_series)}; tabela ponizej pokazalaby jeden z nich dwa razy")

if extraction_order:
    order = sorted(per_series)                      # office, tbbt
    names = [exp.DATASET_NAMES.get(d, d) for d in order]

    def per_unit(dataset, key):
        entry = per_series[dataset]["extraction"].get(key)
        return None if entry is None else 1000 * entry["seconds_per_unit"]

    rows, diverging = [], []
    for entry in extraction_order:
        key = entry["key"]
        times = [per_unit(dataset, key) for dataset in order]
        present = [t for t in times if t is not None]
        # (max - min) / mean, the same spread the repeatability block reports, so
        # a difference BETWEEN series and the jitter WITHIN one are on one scale
        span = ((max(present) - min(present)) / (sum(present) / len(present))
                if len(present) > 1 else None)
        if span is not None and span > SERIES_TOLERANCE:
            diverging.append((entry["component"], span, times))
        cells = [entry["component"], entry["unit"], str(entry["batch"])]
        cells += [amount(t) for t in times]
        cells += [amount(100 * span) + "%" if span is not None else "-"]
        cells += [amount(100 * (per_series[d]["extraction"].get(key) or {}).get("spread"))
                  + "%" if per_series[d]["extraction"].get(key) else "-"
                  for d in order]
        cells += [amount((per_series[d]["extraction"].get(key) or {}).get("load_s"))
                  for d in order]
        rows.append(cells)

    tables.show("Rozbicie kosztu ekstrakcji, osobno na serial",
                ["Komponent", "Jednostka", "Wsad"]
                + [f"{n} [ms/jedn.]" for n in names]
                + ["Rozstep"] + [f"rozrzut {n}" for n in names]
                + [f"ladowanie {n} [s]" for n in names],
                rows, align="lll" + "r" * (len(names) * 3 + 1),
                note="Czas na jednostke, wsad i ladowanie osobno dla kazdego serialu. "
                     "'Rozstep' to (max-min)/srednia miedzy serialami, 'rozrzut' to "
                     "ten sam wskaznik WEWNATRZ jednego przebiegu - jesli sa tego "
                     "samego rzedu, roznica miedzy serialami jest szumem maszyny, a "
                     "nie wlasciwoscia materialu. Ladowanie modelu jest kosztem "
                     "jednorazowym na sesje i nie wchodzi do kolumny min/h. Klatka to "
                     "punkt siatki 1,25 s, okno to 64 klatki zrodlowe, wycinek to "
                     "jedna wykryta twarz.")

    print(f"\nkomponenty o rozstepie ms/jedn. powyzej {100 * SERIES_TOLERANCE:.0f}%:")
    if not diverging:
        print("  zaden - czas na jednostke nie zalezy od serialu")
    for component, span, times in sorted(diverging, key=lambda item: -item[1]):
        pair = " wobec ".join(amount(t) for t in times)
        print(f"  {component:<20}{100 * span:5.1f}%   ({pair} ms/jedn.)")
else:
    print("brak pomiaru ekstrakcji")

## 5. E4-D: detekcja sterowana zapytaniem

Osobny pomiar, [`scripts/measure_e4d_cost.py`](../../scripts/measure_e4d_cost.py).

**Jednostką są tu sekundy, nie milisekundy.** E4-D nie jest sygnałem, tylko etapem punktacji: dekoduje klatki pięćdziesięciu najlepszych fragmentów przy każdym zapytaniu. Wstawienie tej liczby obok wartości rzędu 8 ms pomyliłoby czytelnika o trzy rzędy wielkości, dlatego stoi osobno.

Rozbicie na dekodowanie i detekcję jest istotne, nie kosmetyczne: detekcja jest stabilna, a dekodowanie rozrzucone kilkukrotnie, bo zależy od tego, w ilu odcinkach leżą kandydaci danego zapytania i jak daleko od siebie. Jedna suma ukrywałaby, że własny koszt modelu jest znany dobrze, a cała niepewność siedzi w czytaniu plików wideo.

Podawana jest mediana i zakres obserwacji, bez 95. percentyla: przy próbie kilkudziesięciu zapytań powyżej p95 leżą jedna-dwie obserwacje.

Skrypt mierzy pętlę naturalną, czyli koszt zapytania zadanego samotnie. Prawdziwy etap odwraca kolejność i płaci za dekodowanie raz na odcinek, nie raz na zapytanie.

In [ ]:
# label -> key in the payload; the order the stage runs them in
E4D_PARTS = [
    ("dekodowanie klatek", "decoding"),
    ("detekcja YOLOE", "detection"),
    ("wydobycie fraz", "phrases"),
    ("fuzja i przestawienie", "fusion"),
]


def seconds(ms):
    """Milliseconds of the payload as seconds, at the resolution each one needs."""
    return "-" if ms is None else amount(ms / 1000.0)


found_e4d = newest_with("e4d")
if not found_e4d:
    print("brak pomiaru E4-D - uruchom scripts/measure_e4d_cost.py")
else:
    path, record = found_e4d
    e4d = record["data"]["e4d"]
    print(f"pomiar wczytany z: {path.name}")
    print(f"  material:   {e4d.get('dataset')} / {e4d.get('split')}")
    sample = e4d.get("sample")
    if sample:
        print(f"  PROBA:      {sample['measured']} z {sample['population']} zapytan,"
              f" ziarno {sample['seed']}")
        print(f"              {sample.get('draw', '')}")
    else:
        print(f"  zapytan:    {e4d.get('queries_total')} (calosc, bez probkowania)")
    print(f"  odrzucona rozgrzewka: {e4d.get('warmup_skipped')} pierwsze zapytania")

    whole = e4d.get("total", {})
    rows = []
    for label, key in E4D_PARTS + [("RAZEM", "total")]:
        entry = e4d.get(key) or {}
        if not entry.get("n"):
            continue
        low, high = entry.get("min_ms"), entry.get("max_ms")
        span = f"{seconds(low)} - {seconds(high)}" if low is not None else "-"
        share = (100.0 * entry["total_s"] / whole["total_s"]
                 if whole.get("total_s") else None)
        rows.append([label, entry["n"], seconds(entry["median_ms"]), span,
                     amount(share) + "%" if share is not None else "-"])

    measured_on = exp.DATASET_NAMES.get(e4d.get("dataset"), e4d.get("dataset"))
    tables.show(f"E4-D: koszt obslugi jednego zapytania ({measured_on}, bez usredniania)",
                ["Skladowa", "Obserwacji", "Mediana [s]", "Zakres [s]", "Udzial"],
                rows, align="lrrrr",
                note=f"POMIAR JEDNEGO SERIALU: {measured_on}. W odroznieniu od tabeli "
                     "kosztu komponentow te wiersze NIE sa srednia z obu seriali - "
                     "E4-D zmierzono tylko raz, wiec podpis tabeli w pracy musi to "
                     "powiedziec. "
                     "Sekundy, nie milisekundy - o trzy rzedy wielkosci wiecej niz "
                     "pozostale wiersze kolumny 'Zapytanie'. Bez 95. percentyla: "
                     "przy probie kilkudziesieciu zapytan opiera sie on na jednej "
                     "obserwacji. Udzial liczony z czasu lacznego skladowej, bo "
                     "mediany sie nie sumuja. Mierzone petla NATURALNA - koszt "
                     "zapytania zadanego samotnie; etap w przebiegu odwraca petle "
                     "i placi za dekodowanie raz na odcinek.")

    if (e4d.get("decoding") or {}).get("n") and (e4d.get("detection") or {}).get("n"):
        krotnosc = e4d["decoding"]["median_ms"] / e4d["detection"]["median_ms"]
        print(f"\ndekodowanie jest {amount(krotnosc)} raza drozsze od samej detekcji")